In [ ]:
# Dependencies are pinned in requirements.txt (witwidget + tensorflow are version-fragile).
# !pip install protobuf==3.20.3 witwidget tensorflow catboost --quiet

import numpy as np
import pandas as pd
import tensorflow as tf
from witwidget.notebook.visualization import WitWidget, WitConfigBuilder
import joblib  # For loading trained models
from pathlib import Path
def _find_repo_root():
    for cand in [Path.cwd(), *Path.cwd().resolve().parents]:
        if (cand / 'user_study_materials').is_dir():
            return cand
    return Path.cwd()
REPO_ROOT = _find_repo_root()

# Which dataset/model to load into the What-If Tool: 'animals' or 'aliens'
DATASET = 'animals'
_SUB = {'animals':'animals_hobbies','aliens':'alien_hobbies'}[DATASET]
_CSV = {'animals':'animal_dataset.csv','aliens':'alien_dataset.csv'}[DATASET]
_DATA_DIR = REPO_ROOT/'user_study_materials'/'model_and_data_used_in_experiment'/_SUB
DATASET_CSV=_DATA_DIR/_CSV
MODEL_PKL=_DATA_DIR/'model.pkl'
SCALER_PKL=_DATA_DIR/'scaler.pkl'
LABEL_ENCODER_PKL=_DATA_DIR/'label_encoder.pkl'


In [ ]:
df = pd.read_csv(DATASET_CSV)
df_test = df.copy()
print(f"Dataset loaded from {DATASET_CSV}")


In [ ]:
model = joblib.load(MODEL_PKL)
print(f"Model loaded from {MODEL_PKL}")


In [ ]:
scaler = joblib.load(SCALER_PKL)
print(f"Scaler loaded from {SCALER_PKL}")

feature_columns = [col for col in df.columns if col != 'Class']
X_scaled = scaler.transform(df[feature_columns])
df_scaled = pd.DataFrame(X_scaled, columns=feature_columns)
print("Dataset features scaled.")


In [ ]:
label_encoder = joblib.load(LABEL_ENCODER_PKL)
print(f"Label Encoder loaded from {LABEL_ENCODER_PKL}")

df['Class'] = label_encoder.transform(df['Class'])
print("Class labels encoded.")


In [ ]:
# Ensure feature_columns are correctly defined
feature_columns = [col for col in df.columns if col != "Class"]

# Convert dataset into list format (WIT requires JSON-like structure)
instances = df_scaled.to_dict(orient="records")  # Use scaled dataset

# Define class mapping based on encoded values
class_mapping = {i: label for i, label in enumerate(label_encoder.classes_)}
print("Class Mapping:", class_mapping)

# Debugging: Print test predictions using scaled data
test_prediction = model.predict(df_scaled.iloc[:10])  # Use scaled data
print("Raw Model Predictions:", test_prediction)
print(df_scaled.iloc[:10])  # Show scaled feature values
print(df.head(10))  # Show unscaled data for comparison
print(instances[0])  # Show JSON-like instance structure

if "Class" in df_test.columns:
          inputs_df = df_test.drop(columns=["Class"])
instances = df_test.to_dict(orient="records")
print(type(instances))
print(type(instances[0]))
print(instances[0])

In [ ]:
# Define a wrapper function for WIT
class WitModelWrapper:
    def __init__(self, model, class_mapping, scaler):
        self.model = model
        self.class_mapping = class_mapping
        self.scaler = scaler  # Store scaler for prediction

    def predict(self, inputs):

        if isinstance(inputs, dict):  # Ensure inputs is a list of dictionaries
          inputs = [inputs]

        if not all(isinstance(i, dict) for i in inputs):
          raise ValueError("❌ Invalid input format! Expected a list of dictionaries.")

        # Convert inputs to DataFrame
        inputs_df = pd.DataFrame(inputs)
        # Drop the "Class" column if it exists in inputs
        if "Class" in inputs_df.columns:
          inputs_df = inputs_df.drop(columns=["Class"])

        # Ensure correct column order
        inputs_df = inputs_df[feature_columns]

        # Apply scaling before prediction
        inputs_scaled = self.scaler.transform(inputs_df)
        print("Scaled inputs:", inputs_scaled)

        # Get predictions from the model
        preds = self.model.predict(inputs_scaled)
        # Get model predictions (numeric output required)
        preds = self.model.predict_proba(inputs_scaled)  # ✅ Use predict_proba() instead of predict()


        # Convert predictions back to class names
        #categorical_preds = [self.class_mapping[int(pred)] for pred in preds]

        print("Predictions:", preds)  # Debugging output
        return preds.tolist()  # Return categorical predictions

# Wrap the trained model with scaling for WIT predictions
wrapped_model = WitModelWrapper(model, class_mapping, scaler)

# Debugging: Test wrapped model prediction
print("Test Wrapped Model Prediction:", wrapped_model.predict([instances[0]]))


In [ ]:
# Define feature spec for WIT
feature_spec = {feature: tf.io.FixedLenFeature([], tf.float32) for feature in feature_columns}
feature_spec["Class"] = tf.io.FixedLenFeature([], tf.string)  # Ensure "Class" is a string category

# Define class labels
class_labels = list(class_mapping.values())  # Get class names from mapping

config_builder = WitConfigBuilder(instances) \
    .set_custom_predict_fn(wrapped_model.predict) \
    .set_label_vocab(class_labels) \
    .set_target_feature("Class")


# Launch WIT in the notebook
WitWidget(config_builder)